In [ ]:
# =========================================================
# CARI CHANNEL ID YOUTUBE - NICHE HORROR/BRAINROT ANAK
# Untuk dataset skripsi: klasifikasi video viral (XGBoost vs LightGBM)
# =========================================================

# --- 1. INSTALL LIBRARY ---
!pip install google-api-python-client --quiet

# --- 2. IMPORT & SETUP ---
import pandas as pd
import time
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Masukkan API Key kamu di sini
API_KEY = "(masukkan API KEY kamu di sini)"

youtube = build("youtube", "v3", developerKey=API_KEY)

# --- 3. DAFTAR KEYWORD (CAMPURAN GLOBAL + INDONESIA) ---
# Bisa ditambah/diedit sesuai kebutuhan
KEYWORDS = [
    # GLOBAL (English)
    "sakura school simulator horror story",
    "roblox horror story animation",
    "minecraft horror story scary",
    "horror animation school story",
    "scary animation short story",
    "gta 5 horror story animation",
    "creepy animation horror viral",
    "roblox brainrot horror",
    "minecraft jumpscare horror",
    "horror game story animation",

    # INDONESIA
    "sakura school simulator cerita seram",
    "roblox cerita horor animasi",
    "minecraft cerita horor seram",
    "animasi horor sekolah viral",
    "cerita seram animasi pendek",
]

# Berapa hasil maksimal per keyword (1 page = max 50, kita pakai 25)
MAX_RESULTS_PER_KEYWORD = 25

# Minimal jumlah video yang harus dimiliki channel agar lolos filter
MIN_VIDEO_COUNT = 80

# --- 4. (OPSIONAL) DAFTAR CHANNEL ID YANG SUDAH PUNYA ---
# Supaya tidak muncul dobel di hasil pencarian baru.
# Isi dengan 39 channel ID yang sudah kamu kumpulkan, contoh:
EXISTING_CHANNEL_IDS = set([
    "UChirEQ8Q5iGOMEW3TPOo5_Q",
    "UC4_Ssc1v_P5SrrFjbVG821Q",
    "UCGGwuSrxDFQymf66OdM_-pw",
    "UCvwn9pGVzKmMS6Ssd8R0pog",
    "UCYPSqQRAl-E7SyIAPz-9GDA",
    "UClfw7Mx0Vmh4hDAT0STSohw",
    "UC--QxF30QUdNMP4WpTMe6qg",
    "UCTpb9vZ_xp0_n84lyT-2yqw",
    "UCjJdTnEzXa40iNhxtLc_0Ng",
    "UCcwu2DFi0B9WbLSoqeVH9NQ",
    "UCJ9v-fDARn2Vyl1TX9PfVyA",
    "UCrn4zm6SwhoVgNBMT8J_T4A",
    "UCnzbx11s43-5ncWWwejPVHg",
    "UC98_dcRWqBOF8tGrvRnaWmQ",
    "UC63UrA203WWay62-RA37gCA",
    "UCDP_apI00MkhojUlOFb7gGA",
    "UC4Gumj80pOghE8G7xq4Yxjg",
    "UC4hGmH5sABOA70D4fGb8qNQ",
    "UCg55rJ27zjuqod4Cv0ENqPQ",
    "UCawnfDlDlCLiu1ZTIkygc9w",
    "UCGJ9FqZHdsq1tGSzSyyJpSw",
    "UCcliUkdKk9J_7gdsT3No0Xg",
    "UCFQoE23adk7NSPNhBlanu4w",
    "UCwjojsJdcSNP0UT1R-S6mFA",
    "UCR_J_SntqJh5eXw66d5hJxA",
    "UCFSqAZl3AxxMyaHouB2GKSg",
    "UCc2RL8jvY1rJPuCnJVsIQWw",
    "UCUwP2XkG17aD53fmvYoA6Yw",
    "UCfHi2LIb2D4f1hYzX5de0IA",
    "UCQtoYB7gpftk98WsshIG_fQ",
    "UC7GWXKxiy9_AU9TytEh00eg",
    "UCqkVT_bXF2rTyjdIxapaHbw",
    "UCY_vG2Em4Y4zjOJLmVkiQ1g",
    "UCOTQEQd8Rc_Ny30qhz7K5xA",
    "UC1WSSEbkczMjxJ89TGE58jA",
    "UCFjaPtpDqdMozLOLbcAo7eg",
    "UCs1vLs4_lXRMVytjQLizNdA",
    "UCZhUolzN9vMdkjWrnT9DQ2A",
    "UCqN_7EUWgvARJYXaNRMIGOg",
])

# --- 5. FUNGSI PENCARIAN CHANNEL ---
def search_channels_by_keyword(keyword, max_results=25):
    """
    Mencari channel berdasarkan keyword menggunakan search.list (type=channel).
    Mengembalikan list dict: channel_id, channel_title, keyword.
    """
    results = []
    next_page_token = None
    fetched = 0

    while fetched < max_results:
        try:
            request = youtube.search().list(
                q=keyword,
                type="channel",
                part="snippet",
                maxResults=min(50, max_results - fetched),
                pageToken=next_page_token,
                relevanceLanguage="id",
            )
            response = request.execute()
        except HttpError as e:
            print(f"  [ERROR] Keyword '{keyword}': {e}")
            break

        items = response.get("items", [])
        if not items:
            break

        for item in items:
            channel_id = item["snippet"]["channelId"]
            channel_title = item["snippet"]["channelTitle"]
            results.append({
                "channel_id": channel_id,
                "channel_title": channel_title,
                "keyword_ditemukan": keyword,
            })

        fetched += len(items)
        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.3)  # jaga-jaga rate limit

    return results

# --- 5b. FUNGSI AMBIL VIDEO COUNT (BATCH, HEMAT QUOTA) ---
def get_video_counts(channel_ids):
    """
    Mengambil jumlah video (videoCount) untuk daftar channel_id.
    channels.list bisa menerima sampai 50 ID per request -> 1 unit quota per request.
    Mengembalikan dict: {channel_id: video_count}
    """
    video_counts = {}
    channel_ids = list(set(channel_ids))

    for i in range(0, len(channel_ids), 50):
        batch = channel_ids[i:i + 50]
        try:
            request = youtube.channels().list(
                part="statistics",
                id=",".join(batch),
            )
            response = request.execute()
        except HttpError as e:
            print(f"  [ERROR] Gagal ambil statistics untuk batch: {e}")
            continue

        for item in response.get("items", []):
            cid = item["id"]
            count = int(item["statistics"].get("videoCount", 0))
            video_counts[cid] = count

        time.sleep(0.3)

    return video_counts

# --- 6. JALANKAN PENCARIAN UNTUK SEMUA KEYWORD ---
all_results = []

for kw in KEYWORDS:
    print(f"Mencari: {kw} ...")
    res = search_channels_by_keyword(kw, MAX_RESULTS_PER_KEYWORD)
    print(f"  -> ditemukan {len(res)} channel")
    all_results.extend(res)
    time.sleep(0.5)

print(f"\nTotal hasil mentah (sebelum dedup): {len(all_results)}")

# --- 7. BUAT DATAFRAME & DEDUPLIKASI ---
df = pd.DataFrame(all_results)

if not df.empty:
    # Gabungkan keyword jika channel ditemukan dari banyak keyword
    df = (
        df.groupby(["channel_id", "channel_title"])["keyword_ditemukan"]
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
    )

    # Hilangkan channel yang sudah ada di daftar existing
    if EXISTING_CHANNEL_IDS:
        before = len(df)
        df = df[~df["channel_id"].isin(EXISTING_CHANNEL_IDS)]
        print(f"Dihilangkan {before - len(df)} channel yang sudah ada di daftar existing.")

    df = df.sort_values("channel_title").reset_index(drop=True)

print(f"Total channel UNIK baru ditemukan: {len(df)}")

# --- 8. FILTER CHANNEL BERDASARKAN JUMLAH VIDEO (>= MIN_VIDEO_COUNT) ---
if not df.empty:
    print(f"\nMengambil jumlah video tiap channel (untuk filter >= {MIN_VIDEO_COUNT} video)...")
    video_counts = get_video_counts(df["channel_id"].tolist())

    df["video_count"] = df["channel_id"].map(video_counts)

    # Channel yang gagal diambil datanya (mis. dihapus/private) akan NaN -> dibuang juga
    before_filter = len(df)
    df = df.dropna(subset=["video_count"])
    df["video_count"] = df["video_count"].astype(int)

    df = df[df["video_count"] >= MIN_VIDEO_COUNT]
    print(f"Channel sebelum filter: {before_filter} -> sesudah filter (>= {MIN_VIDEO_COUNT} video): {len(df)}")

    df = df.sort_values("video_count", ascending=False).reset_index(drop=True)

# --- 9. SIMPAN KE CSV ---
output_path = "channel_baru_horror_brainrot.csv"
df.to_csv(output_path, index=False)
print(f"\nDisimpan ke: {output_path}")

# Tampilkan preview
df.head(20)

Mencari: sakura school simulator horror story ...
  -> ditemukan 25 channel
Mencari: roblox horror story animation ...
  -> ditemukan 25 channel
Mencari: minecraft horror story scary ...
  -> ditemukan 25 channel
Mencari: horror animation school story ...
  -> ditemukan 25 channel
Mencari: scary animation short story ...
  -> ditemukan 25 channel
Mencari: gta 5 horror story animation ...
  -> ditemukan 25 channel
Mencari: creepy animation horror viral ...
  -> ditemukan 25 channel
Mencari: roblox brainrot horror ...
  -> ditemukan 25 channel
Mencari: minecraft jumpscare horror ...
  -> ditemukan 25 channel
Mencari: horror game story animation ...
  -> ditemukan 25 channel
Mencari: sakura school simulator cerita seram ...
  -> ditemukan 25 channel
Mencari: roblox cerita horor animasi ...
  -> ditemukan 25 channel
Mencari: minecraft cerita horor seram ...
  -> ditemukan 25 channel
Mencari: animasi horor sekolah viral ...
  -> ditemukan 25 channel
Mencari: cerita seram animasi pendek ...


,channel_id,channel_title,keyword_ditemukan,video_count
0,UCV6mNrW8CrmWtcxWfQXy11g,DarkViperAU,gta 5 horror story animation,5017
1,UCeuTqSaqk32vp1cn8W-8IWg,Bushwhack 18,minecraft jumpscare horror,4921
2,UCMjKNPhYPMegCQ8BqE-UYQA,COCO MEONG,animasi horor sekolah viral,4130
3,UCUG6MpKhlJunvUKt6n_zqrw,KeiKei,roblox cerita horor animasi,4055
4,UCHb7bDmZ5AQJFQrGPntk3fg,Teguh Sugianto,minecraft cerita horor seram,3819
5,UCdZoVLv1XHLLLJTSNcifuDQ,RedHatter,gta 5 horror story animation,3563
6,UCm99c6g9MKBL4lNTlWrjyFQ,Checkpoint,gta 5 horror story animation,3202
7,UC6P2KliQNaLvHMRvGs3bqJg,GamingWithGarry,gta 5 horror story animation,2955
8,UCZBY6V8Lxmwu8gGRBOyO11w,Kubz Scouts,horror game story animation,2923
9,UCe56RtxwepRSyZ-b--SedOw,DangMattSmith,"gta 5 horror story animation, horror game stor...",2769


In [ ]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:


# ============================================================
# YOUTUBE DATA COLLECTOR - SKRIPSI
# Klasifikasi Viral Horror/Brainrot Anak (XGBoost vs LightGBM)
# Google Colab Version
# ============================================================
# OUTPUT KOLOM:
#   video_id, title, description, tags, published_at,
#   duration_seconds, view_count, like_count, comment_count,
#   channel_id, channel_title, channel_subscriber_count,
#   channel_total_views, channel_video_count
# TIDAK ada kolom is_viral — akan dihitung di preprocessing
# ============================================================

# STEP 1: Install & Import
!pip install google-api-python-client pandas tqdm --quiet

import pandas as pd
import time
import random
import re
import json
import os
from datetime import datetime, timezone
from tqdm import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# STEP 2: Konfigurasi
API_KEY = "(masukkan API KEY kamu di sini)"   # ← ganti dengan API key kamu
CUTOFF_DATE = datetime(2026, 1, 1, tzinfo=timezone.utc)   # ambil video publishedAt < 2026-01-01
OLDEST_DATE  = datetime(2020, 1, 1, tzinfo=timezone.utc)  # stop pagination jika video sudah terlalu lama
MAX_VIDEOS_PER_CHANNEL = 300
PROGRESS_FILE = "/content/scraping_progress.json"  # resume jika Colab disconnect

youtube = build("youtube", "v3", developerKey=API_KEY)

# STEP 3: Daftar 200 Channel ID
CHANNEL_IDS = [
    "UCV6mNrW8CrmWtcxWfQXy11g", "UCeuTqSaqk32vp1cn8W-8IWg", "UCMjKNPhYPMegCQ8BqE-UYQA",
    "UCUG6MpKhlJunvUKt6n_zqrw", "UCHb7bDmZ5AQJFQrGPntk3fg", "UCdZoVLv1XHLLLJTSNcifuDQ",
    "UCm99c6g9MKBL4lNTlWrjyFQ", "UC6P2KliQNaLvHMRvGs3bqJg", "UCZBY6V8Lxmwu8gGRBOyO11w",
    "UCe56RtxwepRSyZ-b--SedOw", "UCz_yrc3BO9ve7UgDGZn4dig", "UC7ftRUwi1ATCCEQhD-WPXFg",
    "UCqzxU2_l0r_48-RAy0lBHWg", "UCmaSGxFEpa3LvlavKfvbnMQ", "UC-g_O9HWb5B0rVDG1hLd5Hw",
    "UCVTKO1D_G19UlK03ezaADag", "UCis4Exp9BuIxaroiJqoumuA", "UCRkuUgtDAL4XSU5jB40J_wA",
    "UC-I20klktrosJRJxv9aQhew", "UCqMurYXWqmhxrg-7mv7drFw", "UCCiEKK4835FTejF4ogW4P-A",
    "UCBGp_XPj3Oudk207UNw_tAg", "UCiyvg8v3fJNJNcrh_yQUk3w", "UCO87c90DZllM_mAjrb5irpg",
    "UC7YsDqUlM3Y945tFr2edsAQ", "UCFlp6RS46B4KHuyjuEJVMIQ", "UCfYQlpQc8kTubn_sbDSBrvw",
    "UCpViDMEfvur724mBt8tLiHw", "UCOfyvsftRqM5yyqpT392b4A", "UC6Wl77tvLJCAZDlzq2CWJhg",
    "UCPNoCO-IKY1PIQZCUqf4OfA", "UCRAonprTt59CIOAqXrX0bPg", "UClxdDtMfv9h63-Zp5R81FKw",
    "UC9vKBg57CQ0jjZtaPsXBL6w", "UCMOB6uDg7e-h8OuCw8dK2_Q", "UCRrZWZ8p_Jvhjc8w2KW3EBg",
    "UCp-23W7bSmM_hsfQoWylw7A", "UCt3mMVAyGhwQb_ZBGp_p85g", "UC72VA1LrWHlY86RBbMwGRxw",
    "UCYzEMRKqrh01-tauv7MYyVQ", "UC8Z766VdGCg9sSySmeL-APQ", "UCijMyqnCWgtIUKamfPMvLng",
    "UCpJ6Dn01AqjvFVN2EiK72Ag", "UCr_s2VHtNnGH8LngWlsoZhw", "UC3RYNOKIpKE_2P-5ZapWIYg",
    "UCel2B-6wZhvQHRAKXmpm1Ew", "UCHudxWzwLDAr06TWFmrP7hQ", "UCeI90iCLm4RrNguS4WwNVVQ",
    "UC1cu3C3XRmGee8b1jifMQWA", "UCToxKVrkEuAONR4rFIJ_DyQ", "UCP7k9JoHWrOfbh0slnSOn4g",
    "UCkRNRz9ifB2A04zTb6Y9LYg", "UCbzKhDKRa4YznUc1_f0SBtw", "UCWXcdemE-1OGDhaSAPAw-4w",
    "UCL8_vmF3_0473VvfU7Hi1ug", "UCZVTAKSm7iwy0xX1FyK1VEQ", "UCZvpRu_dNQz1EiFMs4_RNbw",
    "UCVYpWgVdCz0tWUvJmMm25LQ", "UCn4RkjqDN4UE0Uy3y1-K6_w", "UChu9w8GFik0spCfQGF_i9WQ",
    "UCLbvwjbu3Qr0m2EmHHnXC5w", "UCZ9zJNZDmuQQ7Uw97Kdjwkw", "UC9F0hCyH1fa1MrJIfTbSrbw",
    "UCaX76ChBGYNPlcTEGIJSqig", "UCBaJVMx5fcWOHsmPH3IxKKQ", "UCcca5ffHmgRrEwr-9IW-7Uw",
    "UCF3gRq4LbNVVR8QfFUyyTCg", "UCKLyFxoYvEVG-dsELqFlh1Q", "UCm3N3fP3lTgtr2M9svV1sJw",
    "UCF_auq-M9F2Wk4IzIJoHfgg", "UCewhVXvwUtIczThrKiiD-NQ", "UC0XWC2_UZMkXPGn4bj0R2Uw",
    "UCnM02drQP-dF7WMgtJHR4Xw", "UClU1R-QbUDvThzMjMX7NQjQ", "UC261_KcN9A6EIie_Rq0vlWg",
    "UC7h-brAMKgiyrh_BmdhKK3w", "UCiVJZlfcmWMUiXv5oywnDhQ", "UCvCzXDxkH2y32c7VZojgXWA",
    "UCYC3WWrqFERLEnE9jPkRMvw", "UCsCAWoG-SagS7aFuWT1IZPw", "UCMByd3qDySZD1pNmK0ZkmYg",
    "UCZt-N6QkaEnQ4YGDSWdNdPA", "UCgklEUQ8_UR8UY510UFbkxQ", "UChIjAP6RSD0hVWg4fqUdIkA",
    "UCmr_vYKbp98f6kdRoer0iVw", "UCj2Pn8nWcMGnDOdzstvVnbA", "UCph79h9QwbmGwINn45iX5xw",
    "UCQFWbPizb7e8IoM8kvs55cw", "UC7hVpCUJxZD0ACqGQ3zpJ9g", "UCjIctE36xq0SPJoeR8LO5kA",
    "UCRwUO6OIA2DvQA8L1VuzovQ", "UC5IA7MpYcbhbA28RgyQrV0A", "UCOb-ws0S8DmfU4nq4M4z6Rw",
    "UCGkM55uBaAoZziazvvGUlYw", "UC8jqXuCZu7SmpdcNW8hm48Q", "UC6gY0ndnF7q7uTYoZ8Yst5A",
    "UCghiByJ7oj0iu5phWfosLWw", "UCj1g6-RzNx4hvVE4IMzAWuA", "UCUJ0K_e0ac1vBBqss99wudQ",
    "UC8BzE8H7qoFa60JjaRgmIfg", "UCnv6jJ0cdl81j1oWQYbXcsw", "UCaw5JMrP0vgzOHhUKRrxx5g",
    "UCOC5YUqfdbwlnpeMfSTx26Q", "UCVAwRh03m3_1pZjRKvelomA", "UCU0Gd4GEr9rBTIdZC6U3v6w",
    "UCsWwltHud8jAo7txdIOqMSg", "UCCFKfO6XSZ_qEd3kZeOK2Ew", "UC7GZguUCh8gdEs-VKhWqkWA",
    "UC5vY06NlKGpFLTHtMuVoxhw", "UCRu5CV_BaUwJ5QCnunHKvkg", "UCGr3xDf5W2bh0l7o9BQB1oA",
    "UC95VQDMjwvr7GsejfIdS6GQ", "UCquckvYObYDVfMc7ODQ-_WQ", "UCudi3Og-hkRW0HRkiWj2LVw",
    "UCTjF8TNC7blVsBtWyMrsj4A", "UCRnI8mSAO35rx7T4-BFLxtA", "UCS3Iac7JbyHzuXmYj34weRw",
    "UC8b-bA_uLbZHRretA5FK_JQ", "UCyK5hrNdv4293IyUXs19wXw", "UCS8uIXvlSK3HtC0p0VIWyTQ",
    "UCwCKddovSCpQ6PNpIf7ORMA", "UCVZ9t0crMOnaOTz2WyWTmSA", "UCcya9e_wl9ql44-Ee3AETkg",
    "UCM-Aqd9IOe9wBCcAL8LTf7w", "UC33b0wkIKmQm3LzUaQTDGUQ", "UC7rnQz9376zpsBRg8_MmHtQ",
    "UCAj92HhkGw2vgxbdtohJ7QA", "UCtk-l_31BDew-izEFTP1V3Q", "UCnUtxiw-APwRWADsXedTufQ",
    "UCK-21bv_K3pgkDDz_Pzcwnw", "UCjX0ISWJK2lhN3FODrUP0HA", "UC_coP4HpCrkEeBgk3vkxKpQ",
    "UC-RQhtMze3BUzskjv7fumpg", "UCi21QsYAY8kw1YjFq16lZyw", "UCcTKyROPRK964GWtxKJZ10A",
    "UCUIAhYBHAPQTNEyBNl54PRw", "UCR3Mbbg6Ui3vRUAwGFFcqLQ", "UC4lT8b7j4Aovm4TTTnR_RVQ",
    "UCzxEaf1k5OqfERmEaSVi8pQ", "UC02tDemqzG7D4Ep3K4D4Ptw", "UCMYevdNaWmPbuIB0PdPc9ZQ",
    "UCMiLo09ZDt1qS3tCZvtztqw", "UC4pTyJ_o9dSr3nI367GbYyA", "UCKSucZoKBFtsBWvDtq8EBjg",
    "UC_abqMV9cMg-TcQju9kXa5g", "UCGi0NdxX2B36hD6WxXdNhFA", "UCAExz9xtSK8M-dA0IDaYZvg",
    "UCXMwCQVey4PhdWFJ6WhAVhA", "UCbQQ8TCALe4GbyxBeKRuReQ", "UCqykeiGoNyY0Lk784NZ8smQ",
    "UCYD71_aakEU0VT6TwsvV1MA", "UCRrR_UV2aJkGlchEtBxh9gg", "UCn7Ef9w8dTXyk9kNgti9aWw",
    "UCJ87OGZu8iQUO8IyACYYREg", "UCQ2ni-j9Hbtbzt9FytvkBUQ", "UCnVGkEmk7_bbnB8opoerEAQ",
    "UC_z0s6xsJB-BNDaW57wHg2g", "UCuK2rjMid48EA6XoQPe4PIA", "UCHrL9XhIqk7LV9Ya6Qa_Hcg",
    "UCFZvDOxST9vb0nX-Z24YTaw", "UCmsed0Kr1fPl-mvp6iM-_9g", "UCRTsD38mMGi7HnWFzlDUFYA",
    "UCMy725vbeMFUdsFvxTO2zVA", "UCubtXu0Sos902zWt2SfOfTA", "UCuuJUQ8vFxTOUX3X_uK45Dw",
    "UCOdTTViMfSVhc32hmii_Ptg", "UC0p6HfB0vLVMDgvkUCsrpHw", "UCw9gSvWYARIPtYSlVv7b6fA",
    "UC40ed2Fp3t64Wl-7y-zWLcA", "UC3QFis16K3OvRO-3magXLwg", "UCRUoCIu-oho_bnMkhCxyrpg",
    "UC9n3IgleIaFrJrp2hipGQpQ", "UCZX5B1TcHWJKbwAQuYwEh_w", "UCHgTQpnZJY6nW3X61JbtP2Q",
    "UC2zbpOnhs0PwkHza5mkpDYQ", "UC2hm5rD_IrfYRMfq5YQudgA", "UCpZLNDzplh9FDioOX-jnvKA",
    "UC12ClG0_7aPLNdRNNl_4zTA", "UCw-WWK1U5Nlds-ydZNCrkcg", "UCfEd-og-67AUEGwUhcUuzqw",
    "UCFNTq9XKHDNy_1-2lL0kqCg", "UCywlrebEcnkJ6m1y9YViWoQ", "UC_6HKUyjbWkU2iZpnvcJdXA",
    "UCkEKvvUyjcBU4izHBL8Jq4A", "UCCyocdgwZVcIfnWzwhVEdLA", "UCTCdn0QxSosQGnaAPgFrBcQ",
    "UC5zayj7sRZ_zHSqwMmduI4g", "UCVlOEP2q7meEoneD5qgeTxw", "UCIgexLjhSyzvitCywx-m3vw",
    "UCfKQqn93rsJjRa8s2DWLrug", "UCCfaWHyOqKfen_-p0b7L0Xw", "UC8rPkG068jFlejCMBDH582w",
    "UCSH77L4K8TrBgVOEdmF3zxw", "UCBp-eiOdKwtEDQ88wqb8fmg"
]
print(f"Total channel: {len(CHANNEL_IDS)}")

# STEP 4: Helper — parse durasi ISO 8601 ke detik
def parse_duration(duration_iso):
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration_iso or "")
    if not match:
        return 0
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    return h * 3600 + m * 60 + s

def parse_dt(iso_str):
    """Parse ISO 8601 string ke datetime UTC."""
    if not iso_str:
        return None
    return datetime.fromisoformat(iso_str.replace("Z", "+00:00"))

# STEP 5: Load/save progress (resume-safe)
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {"done_channels": [], "rows": []}

def save_progress(done_channels, rows):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"done_channels": done_channels, "rows": rows}, f)

# STEP 6: Ambil data channel + uploads playlist ID (1 API call, bukan 2)
def get_channel_info(channel_id):
    """
    1 call: channels().list(part="snippet,statistics,contentDetails")
    Return: dict info channel + playlist_id uploads
    """
    try:
        resp = youtube.channels().list(
            part="snippet,statistics,contentDetails",
            id=channel_id
        ).execute()

        if not resp.get("items"):
            return None

        item = resp["items"][0]
        stats = item.get("statistics", {})
        playlist_id = (
            item.get("contentDetails", {})
                .get("relatedPlaylists", {})
                .get("uploads", None)
        )
        return {
            "channel_id":                 channel_id,
            "channel_title":              item["snippet"]["title"],
            "channel_subscriber_count":   int(stats.get("subscriberCount", 0)),
            "channel_total_views":        int(stats.get("viewCount", 0)),
            "channel_video_count":        int(stats.get("videoCount", 0)),
            "uploads_playlist_id":        playlist_id,
        }
    except HttpError as e:
        print(f"  [ERROR] channels.list {channel_id}: {e}")
        return None

# STEP 7: Ambil video IDs dari uploads playlist
# Stop paginasi jika sudah lewat OLDEST_DATE (hemat quota)
def get_video_ids_from_playlist(playlist_id, max_videos=300):
    """
    Ambil video IDs dari uploads playlist.
    Stop jika: (a) sudah max_videos, atau (b) video sudah lebih tua dari OLDEST_DATE.
    Return: list of (video_id, published_at_str)
    """
    items_out = []
    next_page = None

    while len(items_out) < max_videos:
        try:
            resp = youtube.playlistItems().list(
                part="contentDetails",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page
            ).execute()
        except HttpError as e:
            print(f"  [ERROR] playlistItems.list: {e}")
            break

        page_items = resp.get("items", [])
        if not page_items:
            break

        stop_early = False
        for item in page_items:
            cd = item.get("contentDetails", {})
            vid_id = cd.get("videoId")
            pub_at = cd.get("videoPublishedAt", "")

            # Cek apakah sudah terlalu lama
            dt = parse_dt(pub_at)
            if dt and dt < OLDEST_DATE:
                stop_early = True
                break

            items_out.append((vid_id, pub_at))
            if len(items_out) >= max_videos:
                stop_early = True
                break

        next_page = resp.get("nextPageToken")
        if stop_early or not next_page:
            break

    return items_out

# STEP 8: Ambil detail video batch (max 50 per call)
def get_video_details_batch(video_id_list, channel_info):
    """
    1 call videos().list untuk maks 50 video ID.
    Filter: publishedAt harus < CUTOFF_DATE (2026-01-01).
    Return: list of row dicts.
    """
    if not video_id_list:
        return []
    try:
        resp = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_id_list)
        ).execute()
    except HttpError as e:
        print(f"  [ERROR] videos.list: {e}")
        return []

    rows = []
    for item in resp.get("items", []):
        snippet    = item.get("snippet", {})
        stats      = item.get("statistics", {})
        content    = item.get("contentDetails", {})

        pub_at = snippet.get("publishedAt", "")
        dt = parse_dt(pub_at)

        # Filter: hanya video sebelum 2026
        if dt and dt >= CUTOFF_DATE:
            continue

        tags = snippet.get("tags", [])

        rows.append({
            # --- identitas video ---
            "video_id":                   item.get("id", ""),
            "title":                      snippet.get("title", ""),
            "description":                snippet.get("description", ""),
            "tags":                       "|".join(tags),   # pakai | bukan , supaya tidak confuse CSV
            "published_at":               pub_at,
            "duration_seconds":           parse_duration(content.get("duration", "")),

            # --- engagement (raw, belum ada label viral) ---
            "view_count":                 int(stats.get("viewCount", 0)),
            "like_count":                 int(stats.get("likeCount", 0)),
            "comment_count":              int(stats.get("commentCount", 0)),

            # --- info channel (denormalized, biar 1 file cukup) ---
            "channel_id":                 channel_info["channel_id"],
            "channel_title":              channel_info["channel_title"],
            "channel_subscriber_count":   channel_info["channel_subscriber_count"],
            "channel_total_views":        channel_info["channel_total_views"],
            "channel_video_count":        channel_info["channel_video_count"],
        })
    return rows

# STEP 9: EKSEKUSI SCRAPING (dengan resume)
print("=" * 65)
print("SCRAPING 200 CHANNELS — HORROR/BRAINROT ANAK")
print(f"Filter: publishedAt < 2026-01-01  |  max {MAX_VIDEOS_PER_CHANNEL} video/channel")
print("=" * 65)

progress    = load_progress()
done_ids    = set(progress["done_channels"])
all_rows    = progress["rows"]
failed      = []
quota_used  = 0

remaining = [ch for ch in CHANNEL_IDS if ch not in done_ids]
print(f"Sudah selesai sebelumnya : {len(done_ids)} channel")
print(f"Sisa yang akan diproses  : {len(remaining)} channel\n")

for idx, channel_id in enumerate(tqdm(remaining, desc="Progress")):
    print(f"\n[{len(done_ids)+1}/{len(CHANNEL_IDS)}] {channel_id}")

    # --- ambil info channel (1 call) ---
    ch_info = get_channel_info(channel_id)
    quota_used += 1
    if not ch_info or not ch_info.get("uploads_playlist_id"):
        print(f"  ✗ Skip (channel tidak ditemukan atau tidak ada uploads playlist)")
        failed.append(channel_id)
        done_ids.add(channel_id)
        save_progress(list(done_ids), all_rows)
        continue

    print(f"  ✓ {ch_info['channel_title']} | {ch_info['channel_subscriber_count']:,} subs")

    # --- ambil video IDs (N calls playlistItems.list) ---
    vid_pairs = get_video_ids_from_playlist(
        ch_info["uploads_playlist_id"],
        max_videos=MAX_VIDEOS_PER_CHANNEL
    )
    pages_used = (len(vid_pairs) // 50) + 1
    quota_used += pages_used

    vid_ids = [v[0] for v in vid_pairs if v[0]]
    print(f"  → {len(vid_ids)} video IDs ditemukan ({pages_used} page playlistItems)")

    if not vid_ids:
        done_ids.add(channel_id)
        save_progress(list(done_ids), all_rows)
        continue

    # --- ambil detail video (batch 50, N calls videos.list) ---
    channel_rows = []
    for i in range(0, len(vid_ids), 50):
        batch = vid_ids[i:i+50]
        rows  = get_video_details_batch(batch, ch_info)
        channel_rows.extend(rows)
        quota_used += 1
        time.sleep(0.3)

    all_rows.extend(channel_rows)
    done_ids.add(channel_id)
    print(f"  ✓ {len(channel_rows)} video disimpan | quota terpakai ~{quota_used} unit")

    # --- simpan progress setiap channel ---
    save_progress(list(done_ids), all_rows)

    # delay antar channel
    time.sleep(random.uniform(1.0, 2.0))

# STEP 10: Buat DataFrame & simpan CSV
print("\n" + "=" * 65)
print("MENYIMPAN DATASET")
print("=" * 65)

df = pd.DataFrame(all_rows)

# Buang duplikat video_id (kalau channel overlap)
before = len(df)
df = df.drop_duplicates(subset="video_id").reset_index(drop=True)
print(f"Duplikat video dihapus: {before - len(df)} baris")

# Urutkan dari terbaru
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df = df.sort_values("published_at", ascending=False).reset_index(drop=True)

OUTPUT_PATH = "/content/dataset_raw_skripsi.csv"
df.to_csv(OUTPUT_PATH, index=False)

print(f"\n✓ File disimpan: {OUTPUT_PATH}")
print(f"  Total baris  : {len(df):,}")
print(f"  Total kolom  : {len(df.columns)}")
print(f"  Kolom        : {list(df.columns)}")
print(f"  Quota terpakai (estimasi): ~{quota_used} / 10,000 unit")
print(f"  Channel berhasil : {len(done_ids) - len(failed)}")
print(f"  Channel gagal    : {len(failed)}")

# STEP 11: Preview & statistik ringkas
print("\n--- Preview 5 baris pertama ---")
display(df.head())

print("\n--- Statistik engagement ---")
print(df[["view_count","like_count","comment_count","duration_seconds"]].describe().round(0))

print("\n--- Distribusi tahun upload ---")
print(df["published_at"].dt.year.value_counts().sort_index())

# STEP 12: Download
from google.colab import files
files.download(OUTPUT_PATH)
print(f"\n⬇ Download dimulai: dataset_raw_skripsi.csv")

if failed:
    fail_path = "/content/failed_channels.txt"
    with open(fail_path, "w") as f:
        f.write("\n".join(failed))
    files.download(fail_path)
    print(f"⬇ Download dimulai: failed_channels.txt ({len(failed)} channel gagal)")

print("\nSelesai.")

Total channel: 194
SCRAPING 200 CHANNELS — HORROR/BRAINROT ANAK
Filter: publishedAt < 2026-01-01  |  max 300 video/channel
Sudah selesai sebelumnya : 0 channel
Sisa yang akan diproses  : 194 channel



Progress:   0%|          | 0/194 [00:00<?, ?it/s]


[1/194] UCV6mNrW8CrmWtcxWfQXy11g
  ✓ DarkViperAU | 2,210,000 subs
  → 300 video IDs ditemukan (7 page playlistItems)
  ✓ 0 video disimpan | quota terpakai ~14 unit


Progress:   1%|          | 1/194 [00:06<21:43,  6.75s/it]


[2/194] UCeuTqSaqk32vp1cn8W-8IWg
  ✓ Bushwhack 18 | 3,000,000 subs
  → 300 video IDs ditemukan (7 page playlistItems)
  ✓ 0 video disimpan | quota terpakai ~28 unit


...

,video_id,title,description,tags,published_at,duration_seconds,view_count,like_count,comment_count,channel_id,channel_title,channel_subscriber_count,channel_total_views,channel_video_count
0,KLNViIu9n1U,Finding Every CAPYBARA in Roblox Find The Capy...,"Today, Johnny finds all of the capybaras in Ro...",Digging to Escape from Roblox Prison...|Playin...,2025-12-31 23:47:51+00:00,1170,131697,1275,114,UCHudxWzwLDAr06TWFmrP7hQ,Johnny and Marty,9980000,1595235224,928
1,Zb_0l50TJHA,Working at a SCARY FOOD TRUCK in Roblox!,We just dropped NEW MERCH! Grab yours before i...,Digging to Escape from Roblox Prison...|Playin...,2025-12-31 22:57:35+00:00,969,292716,3007,391,UCHudxWzwLDAr06TWFmrP7hQ,Johnny and Marty,9980000,1595235224,928
2,ehfhV-NCkSE,Top 5 Avatar,what top 5 do you want next tho?\n------------...,top 5|dog|dogs|top 5 dog|top 5 dogs|dog meme|d...,2025-12-31 22:57:03+00:00,54,48329,3507,215,UCWXcdemE-1OGDhaSAPAw-4w,squewe,2340000,586989476,812
3,C2Rzk_57ibo,Top 10 Scary Cartoon Theories That Will RUIN Y...,Top 10 Scary Cartoon Theories That Will Ruin Y...,10|top 10|most amazing top 10|top 10 scary|dis...,2025-12-31 22:32:01+00:00,627,24811,1169,169,UCe56RtxwepRSyZ-b--SedOw,DangMattSmith,15200000,3597768054,2770
4,UAbjomUmYnI,Do Not Answer The Doorbell Tonight | VHS Analo...,Become a member: https://www.youtube.com/chann...,texting|text story|horror texts|texts|text con...,2025-12-31 22:15:01+00:00,2265,88903,1633,403,UCeI90iCLm4RrNguS4WwNVVQ,Textify,475000,137363365,865



--- Statistik engagement ---
        view_count  like_count  comment_count  duration_seconds
count      33339.0     33339.0        33339.0           33339.0
mean     1157655.0     23042.0          875.0            1329.0
std      3799417.0     91844.0         3013.0            2597.0
min            0.0         0.0            0.0               0.0
25%        34876.0       758.0           41.0             136.0
50%       163784.0      3236.0          178.0             773.0
75%       765560.0     13458.0          656.0            1431.0
max    186642602.0   3829569.0       167056.0           64163.0

--- Distribusi tahun upload ---
published_at
2020     1196
2021     2183
2022     3003
2023     4986
2024     7395
2025    14576
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇ Download dimulai: dataset_raw_skripsi.csv

Selesai.


In [ ]:
from google.colab import files
files.download('/content/dataset_raw_skripsi.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# YOUTUBE DATA COLLECTOR - SKRIPSI
# Klasifikasi Viral Horror/Brainrot Anak (XGBoost vs LightGBM)
# Google Colab Version
# ============================================================
# OUTPUT KOLOM:
#   video_id, title, description, tags, published_at,
#   duration_seconds, view_count, like_count, comment_count,
#   channel_id, channel_title, channel_subscriber_count,
#   channel_total_views, channel_video_count
# TIDAK ada kolom is_viral — akan dihitung di preprocessing
# ============================================================

# STEP 1: Install & Import
!pip install google-api-python-client pandas tqdm --quiet

import pandas as pd
import time
import random
import re
import json
import os
from datetime import datetime, timezone
from tqdm import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# STEP 2: Konfigurasi
API_KEY = "(masukkan API KEY kamu di sini)"   # ← ganti dengan API key kamu
CUTOFF_DATE = datetime(2026, 1, 1, tzinfo=timezone.utc)   # ambil video publishedAt < 2026-01-01
OLDEST_DATE  = datetime(2020, 1, 1, tzinfo=timezone.utc)  # stop pagination jika video sudah terlalu lama
MAX_VIDEOS_PER_CHANNEL = 300
PROGRESS_FILE = "/content/scraping_progress_batch2.json"  # resume jika Colab disconnect

youtube = build("youtube", "v3", developerKey=API_KEY)

# STEP 3: Daftar 38 Channel ID Lama (batch 2)
# Catatan: UCawnfDlDlCLiu1ZTIkygc9w muncul 2x di list asli → sudah dedup jadi 38 unik
CHANNEL_IDS = [
    "UChirEQ8Q5iGOMEW3TPOo5_Q", "UC4_Ssc1v_P5SrrFjbVG821Q", "UCGGwuSrxDFQymf66OdM_-pw",
    "UCvwn9pGVzKmMS6Ssd8R0pog", "UCYPSqQRAl-E7SyIAPz-9GDA", "UClfw7Mx0Vmh4hDAT0STSohw",
    "UC--QxF30QUdNMP4WpTMe6qg", "UCTpb9vZ_xp0_n84lyT-2yqw", "UCjJdTnEzXa40iNhxtLc_0Ng",
    "UCcwu2DFi0B9WbLSoqeVH9NQ", "UCJ9v-fDARn2Vyl1TX9PfVyA", "UCrn4zm6SwhoVgNBMT8J_T4A",
    "UCnzbx11s43-5ncWWwejPVHg", "UC98_dcRWqBOF8tGrvRnaWmQ", "UC63UrA203WWay62-RA37gCA",
    "UCDP_apI00MkhojUlOFb7gGA", "UC4Gumj80pOghE8G7xq4Yxjg", "UC4hGmH5sABOA70D4fGb8qNQ",
    "UCg55rJ27zjuqod4Cv0ENqPQ", "UCawnfDlDlCLiu1ZTIkygc9w", "UCGJ9FqZHdsq1tGSzSyyJpSw",
    "UCcliUkdKk9J_7gdsT3No0Xg", "UCFQoE23adk7NSPNhBlanu4w", "UCwjojsJdcSNP0UT1R-S6mFA",
    "UCR_J_SntqJh5eXw66d5hJxA", "UCFSqAZl3AxxMyaHouB2GKSg", "UCc2RL8jvY1rJPuCnJVsIQWw",
    "UCUwP2XkG17aD53fmvYoA6Yw", "UCfHi2LIb2D4f1hYzX5de0IA", "UCQtoYB7gpftk98WsshIG_fQ",
    "UC7GWXKxiy9_AU9TytEh00eg", "UCqkVT_bXF2rTyjdIxapaHbw", "UCY_vG2Em4Y4zjOJLmVkiQ1g",
    "UCOTQEQd8Rc_Ny30qhz7K5xA", "UC1WSSEbkczMjxJ89TGE58jA", "UCFjaPtpDqdMozLOLbcAo7eg",
    "UCs1vLs4_lXRMVytjQLizNdA", "UCZhUolzN9vMdkjWrnT9DQ2A",
]
print(f"Total channel: {len(CHANNEL_IDS)}")

# STEP 4: Helper — parse durasi ISO 8601 ke detik
def parse_duration(duration_iso):
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration_iso or "")
    if not match:
        return 0
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    return h * 3600 + m * 60 + s

def parse_dt(iso_str):
    """Parse ISO 8601 string ke datetime UTC."""
    if not iso_str:
        return None
    return datetime.fromisoformat(iso_str.replace("Z", "+00:00"))

# STEP 5: Load/save progress (resume-safe)
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {"done_channels": [], "rows": []}

def save_progress(done_channels, rows):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"done_channels": done_channels, "rows": rows}, f)

# STEP 6: Ambil data channel + uploads playlist ID (1 API call, bukan 2)
def get_channel_info(channel_id):
    """
    1 call: channels().list(part="snippet,statistics,contentDetails")
    Return: dict info channel + playlist_id uploads
    """
    try:
        resp = youtube.channels().list(
            part="snippet,statistics,contentDetails",
            id=channel_id
        ).execute()

        if not resp.get("items"):
            return None

        item = resp["items"][0]
        stats = item.get("statistics", {})
        playlist_id = (
            item.get("contentDetails", {})
                .get("relatedPlaylists", {})
                .get("uploads", None)
        )
        return {
            "channel_id":                 channel_id,
            "channel_title":              item["snippet"]["title"],
            "channel_subscriber_count":   int(stats.get("subscriberCount", 0)),
            "channel_total_views":        int(stats.get("viewCount", 0)),
            "channel_video_count":        int(stats.get("videoCount", 0)),
            "uploads_playlist_id":        playlist_id,
        }
    except HttpError as e:
        print(f"  [ERROR] channels.list {channel_id}: {e}")
        return None

# STEP 7: Ambil video IDs dari uploads playlist
# Stop paginasi jika sudah lewat OLDEST_DATE (hemat quota)
def get_video_ids_from_playlist(playlist_id, max_videos=300):
    """
    Ambil video IDs dari uploads playlist.
    Stop jika: (a) sudah max_videos, atau (b) video sudah lebih tua dari OLDEST_DATE.
    Return: list of (video_id, published_at_str)
    """
    items_out = []
    next_page = None

    while len(items_out) < max_videos:
        try:
            resp = youtube.playlistItems().list(
                part="contentDetails",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page
            ).execute()
        except HttpError as e:
            print(f"  [ERROR] playlistItems.list: {e}")
            break

        page_items = resp.get("items", [])
        if not page_items:
            break

        stop_early = False
        for item in page_items:
            cd = item.get("contentDetails", {})
            vid_id = cd.get("videoId")
            pub_at = cd.get("videoPublishedAt", "")

            # Cek apakah sudah terlalu lama
            dt = parse_dt(pub_at)
            if dt and dt < OLDEST_DATE:
                stop_early = True
                break

            items_out.append((vid_id, pub_at))
            if len(items_out) >= max_videos:
                stop_early = True
                break

        next_page = resp.get("nextPageToken")
        if stop_early or not next_page:
            break

    return items_out

# STEP 8: Ambil detail video batch (max 50 per call)
def get_video_details_batch(video_id_list, channel_info):
    """
    1 call videos().list untuk maks 50 video ID.
    Filter: publishedAt harus < CUTOFF_DATE (2026-01-01).
    Return: list of row dicts.
    """
    if not video_id_list:
        return []
    try:
        resp = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_id_list)
        ).execute()
    except HttpError as e:
        print(f"  [ERROR] videos.list: {e}")
        return []

    rows = []
    for item in resp.get("items", []):
        snippet    = item.get("snippet", {})
        stats      = item.get("statistics", {})
        content    = item.get("contentDetails", {})

        pub_at = snippet.get("publishedAt", "")
        dt = parse_dt(pub_at)

        # Filter: hanya video sebelum 2026
        if dt and dt >= CUTOFF_DATE:
            continue

        tags = snippet.get("tags", [])

        rows.append({
            # --- identitas video ---
            "video_id":                   item.get("id", ""),
            "title":                      snippet.get("title", ""),
            "description":                snippet.get("description", ""),
            "tags":                       "|".join(tags),   # pakai | bukan , supaya tidak confuse CSV
            "published_at":               pub_at,
            "duration_seconds":           parse_duration(content.get("duration", "")),

            # --- engagement (raw, belum ada label viral) ---
            "view_count":                 int(stats.get("viewCount", 0)),
            "like_count":                 int(stats.get("likeCount", 0)),
            "comment_count":              int(stats.get("commentCount", 0)),

            # --- info channel (denormalized, biar 1 file cukup) ---
            "channel_id":                 channel_info["channel_id"],
            "channel_title":              channel_info["channel_title"],
            "channel_subscriber_count":   channel_info["channel_subscriber_count"],
            "channel_total_views":        channel_info["channel_total_views"],
            "channel_video_count":        channel_info["channel_video_count"],
        })
    return rows

# STEP 9: EKSEKUSI SCRAPING (dengan resume)
print("=" * 65)
print("SCRAPING 200 CHANNELS — HORROR/BRAINROT ANAK")
print(f"Filter: publishedAt < 2026-01-01  |  max {MAX_VIDEOS_PER_CHANNEL} video/channel")
print("=" * 65)

progress    = load_progress()
done_ids    = set(progress["done_channels"])
all_rows    = progress["rows"]
failed      = []
quota_used  = 0

remaining = [ch for ch in CHANNEL_IDS if ch not in done_ids]
print(f"Sudah selesai sebelumnya : {len(done_ids)} channel")
print(f"Sisa yang akan diproses  : {len(remaining)} channel\n")

for idx, channel_id in enumerate(tqdm(remaining, desc="Progress")):
    print(f"\n[{len(done_ids)+1}/{len(CHANNEL_IDS)}] {channel_id}")

    # --- ambil info channel (1 call) ---
    ch_info = get_channel_info(channel_id)
    quota_used += 1
    if not ch_info or not ch_info.get("uploads_playlist_id"):
        print(f"  ✗ Skip (channel tidak ditemukan atau tidak ada uploads playlist)")
        failed.append(channel_id)
        done_ids.add(channel_id)
        save_progress(list(done_ids), all_rows)
        continue

    print(f"  ✓ {ch_info['channel_title']} | {ch_info['channel_subscriber_count']:,} subs")

    # --- ambil video IDs (N calls playlistItems.list) ---
    vid_pairs = get_video_ids_from_playlist(
        ch_info["uploads_playlist_id"],
        max_videos=MAX_VIDEOS_PER_CHANNEL
    )
    pages_used = (len(vid_pairs) // 50) + 1
    quota_used += pages_used

    vid_ids = [v[0] for v in vid_pairs if v[0]]
    print(f"  → {len(vid_ids)} video IDs ditemukan ({pages_used} page playlistItems)")

    if not vid_ids:
        done_ids.add(channel_id)
        save_progress(list(done_ids), all_rows)
        continue

    # --- ambil detail video (batch 50, N calls videos.list) ---
    channel_rows = []
    for i in range(0, len(vid_ids), 50):
        batch = vid_ids[i:i+50]
        rows  = get_video_details_batch(batch, ch_info)
        channel_rows.extend(rows)
        quota_used += 1
        time.sleep(0.3)

    all_rows.extend(channel_rows)
    done_ids.add(channel_id)
    print(f"  ✓ {len(channel_rows)} video disimpan | quota terpakai ~{quota_used} unit")

    # --- simpan progress setiap channel ---
    save_progress(list(done_ids), all_rows)

    # delay antar channel
    time.sleep(random.uniform(1.0, 2.0))

# STEP 10: Buat DataFrame & simpan CSV
print("\n" + "=" * 65)
print("MENYIMPAN DATASET")
print("=" * 65)

df = pd.DataFrame(all_rows)

# Buang duplikat video_id (kalau channel overlap)
before = len(df)
df = df.drop_duplicates(subset="video_id").reset_index(drop=True)
print(f"Duplikat video dihapus: {before - len(df)} baris")

# Urutkan dari terbaru
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df = df.sort_values("published_at", ascending=False).reset_index(drop=True)

OUTPUT_PATH = "/content/dataset_raw_skripsi_batch2.csv"
df.to_csv(OUTPUT_PATH, index=False)

print(f"\n✓ File disimpan: {OUTPUT_PATH}")
print(f"  Total baris  : {len(df):,}")
print(f"  Total kolom  : {len(df.columns)}")
print(f"  Kolom        : {list(df.columns)}")
print(f"  Quota terpakai (estimasi): ~{quota_used} / 10,000 unit")
print(f"  Channel berhasil : {len(done_ids) - len(failed)}")
print(f"  Channel gagal    : {len(failed)}")

# STEP 11: Preview & statistik ringkas
print("\n--- Preview 5 baris pertama ---")
display(df.head())

print("\n--- Statistik engagement ---")
print(df[["view_count","like_count","comment_count","duration_seconds"]].describe().round(0))

print("\n--- Distribusi tahun upload ---")
print(df["published_at"].dt.year.value_counts().sort_index())

# STEP 12: Download
from google.colab import files
files.download(OUTPUT_PATH)
print(f"\n⬇ Download dimulai: dataset_raw_skripsi_batch2.csv")

if failed:
    fail_path = "/content/failed_channels.txt"
    with open(fail_path, "w") as f:
        f.write("\n".join(failed))
    files.download(fail_path)
    print(f"⬇ Download dimulai: failed_channels.txt ({len(failed)} channel gagal)")

print("\nSelesai.")

Total channel: 38
SCRAPING 200 CHANNELS — HORROR/BRAINROT ANAK
Filter: publishedAt < 2026-01-01  |  max 300 video/channel
Sudah selesai sebelumnya : 0 channel
Sisa yang akan diproses  : 38 channel



Progress:   0%|          | 0/38 [00:00<?, ?it/s]


[1/38] UChirEQ8Q5iGOMEW3TPOo5_Q
  ✓ Dunia Budi | 2,200,000 subs
  → 300 video IDs ditemukan (7 page playlistItems)
  ✓ 161 video disimpan | quota terpakai ~14 unit


Progress:   3%|▎         | 1/38 [00:05<03:33,  5.78s/it]


[2/38] UC4_Ssc1v_P5SrrFjbVG821Q
  ✓ Adam B | 5,440,000 subs
  → 300 video IDs ditemukan (7 page playlistItems)
  ✓ 263 video disimpan | quota terpakai ~28 unit


...

,video_id,title,description,tags,published_at,duration_seconds,view_count,like_count,comment_count,channel_id,channel_title,channel_subscriber_count,channel_total_views,channel_video_count
0,n1Q3t5R1B80,Yuta Mio Dan Baby Celine Jadi Kucing Hancurkan...,#roblox #duniabudi,,2025-12-31 23:01:18+00:00,1555,2049354,21853,419,UCJ9v-fDARn2Vyl1TX9PfVyA,Dunia Budi Roblox,819000,460555023,515
1,fHnrSZDHWY8,Yuta Mio Panik CCTV Rekam ANOMALI SUSU MILO KO...,Bantu LIKE SHARE & SUBSCRIBE supaya YUTA SEMAN...,Sakura|Sakura school simulator|Sakura horror s...,2025-12-31 23:01:06+00:00,608,8524,14,2,UC63UrA203WWay62-RA37gCA,Dunia Kelvin Sakura,28000,14941366,178
2,PtRdpfqhuwI,Yuta Mio Pawai Bus Boboiboy BLAZE ICE DURI Di ...,"Sakura School Simulator\nSelamat Menonton ya, ...",,2025-12-31 23:00:43+00:00,1149,734936,2887,68,UChirEQ8Q5iGOMEW3TPOo5_Q,Dunia Budi,2200000,826523720,1548
3,MMV6EWeO3ZQ,Yuta Mio Buat Rumah Pohon Paling Aman Dari PAS...,Yuta Mio Buat Rumah Pohon Paling Aman Dari PAS...,#sakuraschoolsimulatordrama|#idsakura|#sakuras...,2025-12-31 23:00:30+00:00,829,5308288,10200,142,UCs1vLs4_lXRMVytjQLizNdA,SPONSIX,1040000,513219878,286
4,Jz4ZtUtPWg0,The Craziest Things I've Ever Done,Download Brawl Stars and start creating your o...,,2025-12-31 21:47:31+00:00,37,96437,1646,58,UCR_J_SntqJh5eXw66d5hJxA,Matthew Beem,8100000,1492841088,289



--- Statistik engagement ---
        view_count  like_count  comment_count  duration_seconds
count       5624.0      5624.0         5624.0            5624.0
mean     2073888.0     25397.0          733.0             992.0
std      6544096.0     99989.0         2261.0            1579.0
min            0.0         0.0            0.0               0.0
25%       100186.0       722.0           24.0             486.0
50%       596776.0      3825.0          105.0             661.0
75%      1711267.0     15562.0          398.0            1175.0
max    154934440.0   2631792.0        73401.0           42898.0

--- Distribusi tahun upload ---
published_at
2020     130
2021     168
2022      83
2023     396
2024    1029
2025    3818
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇ Download dimulai: dataset_raw_skripsi_batch2.csv

Selesai.


In [ ]:
# ============================================================
# GABUNGKAN DUA HASIL SCRAPING
# Jalankan di Colab setelah kedua scraping selesai
# ============================================================

import pandas as pd
from google.colab import files

# --- Load dua file CSV ---
# Sesuaikan nama file jika kamu rename salah satunya
df_200 = pd.read_csv("/content/dataset_raw_skripsi.csv")       # hasil 200 channel baru
df_39  = pd.read_csv("/content/dataset_raw_skripsi_batch2.csv")  # hasil 39 channel lama
# Catatan: sebelum run ini, rename dulu salah satu file di Colab
# supaya tidak tertimpa. Klik kanan file di panel Files → Rename.

print(f"Dataset 200 channel : {len(df_200):,} baris")
print(f"Dataset 39 channel  : {len(df_39):,} baris")

# --- Gabungkan ---
df_all = pd.concat([df_200, df_39], ignore_index=True)

# --- Buang duplikat video_id (kalau ada channel overlap antara dua batch) ---
before = len(df_all)
df_all = df_all.drop_duplicates(subset="video_id").reset_index(drop=True)
print(f"Duplikat dihapus    : {before - len(df_all)} baris")

# --- Urutkan dari terbaru ---
df_all["published_at"] = pd.to_datetime(df_all["published_at"], utc=True)
df_all = df_all.sort_values("published_at", ascending=False).reset_index(drop=True)

# --- Simpan ---
OUTPUT = "/content/dataset_gabungan_final.csv"
df_all.to_csv(OUTPUT, index=False)

print(f"\nTotal baris gabungan: {len(df_all):,}")
print(f"Total kolom         : {len(df_all.columns)}")
print(f"Distribusi tahun    :")
print(df_all["published_at"].dt.year.value_counts().sort_index())
print(f"\nDisimpan ke: {OUTPUT}")

# --- Download ---
files.download(OUTPUT)
print("Download dimulai: dataset_gabungan_final.csv")

Dataset 200 channel : 33,339 baris
Dataset 39 channel  : 5,624 baris
Duplikat dihapus    : 0 baris

Total baris gabungan: 38,963
Total kolom         : 14
Distribusi tahun    :
published_at
2020     1326
2021     2351
2022     3086
2023     5382
2024     8424
2025    18394
Name: count, dtype: int64

Disimpan ke: /content/dataset_gabungan_final.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download dimulai: dataset_gabungan_final.csv
